In [ ]:
# General imports
import os
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

# Astropy imports
import astropy.cosmology as Cosmology
from astropy import units as u
from astropy import constants as const
from astropy.coordinates import SkyCoord

# Own-code imports
from repo_root import *
from helper_functions.analytic_neutrino_flux import *

### Used GOALS data

## Herschel IR luminosity data

In [ ]:
file_path = os.path.join(repo_root, "data", "split-lir", "goals_herschel_sample_list.txt")
Herschel = np.loadtxt(file_path)

Split LIR values: https://goals.ipac.caltech.edu/data_files/Lir_LirSurfaceDensity.dat -> IDs missing: 45, 46 , 78, 83, 110, 150, 157,  159 ,187, and 189 + 92 & 238 NaN values. This results in LIR values for 229 galaxies. Note the GOALS sample consists of 202 objects. See also the paper related to the analysis: https://ui.adsabs.harvard.edu/abs/2017ApJ...846...32D/abstract
 
Names corresponding to the iDS in the above file: https://goals.ipac.caltech.edu/data_files/goals_herschel_pacs_CII158_linecatalog_HIPEv13.dat

The redshift and luminosity distance values can be found in "/Users/yarno/documents/PhD/GOALS/goals_herschel_sample_list.txt"

In [ ]:
file_path = os.path.join(repo_root, "data", "split-lir", "lir_split.txt")
LIR_split_array = np.loadtxt(file_path)

ID = LIR_split_array[:, 0].astype(int)
LIR_split = LIR_split_array[:, 1]
LIR_split_unc = LIR_split_array[:, 2]
SigmaIR = LIR_split_array[:, 3]

# Replace NaN surface densities with 0 (vectorised, replaces the old manual loop)
SigmaIR_noNaN = np.nan_to_num(SigmaIR, nan=0.0)

# Classification by IR luminosity regime (in units of 1e11 Lsun)
LIR_irg = LIR_split[LIR_split < 1]
LIR_lirg = LIR_split[(LIR_split >= 1) & (LIR_split < 10)]
LIR_ulirg = LIR_split[LIR_split >= 10]

## Individual  $\langle \alpha_{\mathrm{AGN}} \rangle$ distribution

In [ ]:
file_path = os.path.join(repo_root, "data", "split-lir", "agn_split.txt")
AGN_split_array = np.loadtxt(file_path)

agn_fracs = AGN_split_array[:, 0]
agn_fracs_unc = AGN_split_array[:, 1]
median_agn_frac = np.median(agn_fracs)

The AGN values can be found in Table 2 of this article: https://iopscience.iop.org/article/10.3847/1538-4357/aa81d7#apjaa81d7t1 . It is noted that the IDS 45, 46 , 78, 83, 110, 150, 157,  159 ,187 and 189 are already missing and therefore 92 & 238 had to be removed.


Note, that the mid-infrared and bolometric AGN fractions are both derived from the Spitzer low-res spectra, and are therefore representative of the projected physical area covered by the IRS short-low slit, centered on the nucleus. For more distant or point-like GOALS sources (D > ~ 50-100 Mpc), the AGN fractions will be representative of the values for the entire galaxy.  However, for more nearby sources, the true global mid-infrared and bolometric AGN fractions can be significantly smaller than those reported here. For example, the source with an average AGN fraction of one is NGC 1068, the most nearby (~16 Mpc) Seyfert II galaxy. The true global bolometric AGN fraction will therefore, most likely, be lower.

# Constructing a general dataframe

In [ ]:
def SNr(LIR):
    """Supernova rate from LIR using the Mattila et al. empirical calibration."""
    return 2.7e-12 * LIR


# 33 GHz radio-continuum -> SFR calibration coefficients (see markdown note below)
_SFR_COEFFS = {
    "Murphy": 3.15e-44,
    "Yarno NK": 4.934702e-44,
    "Yarno TH": 1.537522e-44,
}

# Supernova-rate-per-IMF calibration coefficients (see markdown note below)
_SNR_IMF_COEFFS = {
    "Murphy": (1 / 86.3) * 3.88e-44,
    # "Yarno NK": 5.428172e-46,  # non-standard SB99 param (BHs formed from 40 Msun); kept for reference only
    "Yarno NK": 5.973941e-46,
    "Yarno TH": 4.151310e-46,
}


def SFR(LIR, calib):
    """Star-formation rate from LIR [erg/s] for a given calibration name."""
    return _SFR_COEFFS[calib] * LIR * 1e7 * 3.828e26


def SNr_IMF(LIR, calib):
    """IMF-scaled supernova rate from LIR [erg/s] for a given calibration name."""
    return _SNR_IMF_COEFFS[calib] * LIR * 1e7 * 3.828e26

Important note: The commented calibration factor under "Yarno NK" assumes a non-standard SB99 simulation parameter. Specifically, the commented calibration factor takes into account that black holes are already formed from 40 stellar masses. The one that is used for further calculations takes the standard SB99 parameters.

The callibration for the SFR is based on 33GHz continuum data for 56 nuclei and 62 extranuclear regions for star-forming galaxies covering a wide range of integrated properties, ISM conditions, morphological types, IR luminosity range, and and star-formation rates. The Median value for the ratio between the 33GHz star-formation rate and the IR luminosity are consistent for nuclear and extra-nuclear regions. This empirically determined coefficient is consistent with a theoretical relation given in Murpy et al (2011, SFR = 3.88e-44L$_{IR}$,https://ui.adsabs.harvard.edu/abs/2011ApJ...737...67M/abstract and https://ui.adsabs.harvard.edu/abs/2012ApJ...761...97M/abstract).

In [ ]:
# IDs excluded throughout the analysis (see markdown notes above):
# 45, 46, 78, 83, 110, 150, 157, 159, 187, 189 are missing Herschel/ids.txt
# entries; 92 and 238 are dropped for having NaN split-LIR values.
MISSING_IDS = {45, 46, 78, 83, 92, 110, 150, 157, 159, 187, 189, 238}

# --- Name / RA / Dec, keyed by galaxy ID (columns 0, 1, 6, 7 of ids.txt) ---
ids_path = os.path.join(repo_root, "data", "split-lir", "ids.txt")
with open(ids_path, "r") as f:
    id_rows = [line.split() for line in f]

names = [row[1] for row in id_rows if int(row[0]) not in MISSING_IDS]
ras = [row[6] for row in id_rows if int(row[0]) not in MISSING_IDS]
decs = [row[7] for row in id_rows if int(row[0]) not in MISSING_IDS]

# --- Redshift and luminosity distance (columns 3, 4 of the Herschel file) ---
redshifts = [row[3] for row in Herschel if int(row[0]) not in MISSING_IDS]
distances_Mpc = [row[4] for row in Herschel if int(row[0]) not in MISSING_IDS]

# NOTE: names/ras/decs/redshifts/distances_Mpc and the LIR/AGN arrays below are
# assumed to already be in the same galaxy order once IDs 92 and 238 are
# excluded (this matches the assumption made in the original notebook).
records = []
for i in range(len(ID)):
    if ID[i] in (92, 238):
        continue
    log_lir = round(np.log10(LIR_split[i] * 1e11), 2)
    records.append({
        "ID": ID[i],
        "log(LIR)": log_lir,
        "LIR_unc x 1e11": round(LIR_split_unc[i], 2),
        "AGNbol": agn_fracs[i],
        "AGNbol_unc": agn_fracs_unc[i],
        "SFR": round(SFR((1 - agn_fracs[i]) * LIR_split[i] * 1e11, "Yarno NK"), 2),
        "un-corr SFR": round(SFR(LIR_split[i] * 1e11, "Yarno NK"), 2),
        "SigmaIR": SigmaIR[i],
    })

for rec, name, ra, dec, z, d_l in zip(records, names, ras, decs, redshifts, distances_Mpc):
    rec["Name"] = name
    rec["RA"] = ra
    rec["Dec"] = dec
    rec["Redshift"] = z
    rec["D_L [Mpc]"] = d_l

(!) It should be noted that the neutrino flux predictions already take into account that G=0.5.

In [ ]:
# Neutrino-flux model parameters
E = 1e3        # reference cosmic-ray energy scale
n_ism = 1000   # ISM density
R = 250        # region radius
v = 500        # wind/shock velocity
p_max = 1e8    # maximum cosmic-ray momentum
H = 150        # scale height
G = 0.5        # global normalisation factor (already folded into the flux predictions, see note above)

df = pd.DataFrame(
    {
        "Name": [r["Name"] for r in records],
        "RA": [r["RA"] for r in records],
        "Dec": [r["Dec"] for r in records],
        "Redshift": [r["Redshift"] for r in records],
        "D_L [Mpc]": [r["D_L [Mpc]"] for r in records],
        "log(LIR)": [r["log(LIR)"] for r in records],
        "LIR_unc x 1e11": [r["LIR_unc x 1e11"] for r in records],
        "AGNbol": [r["AGNbol"] for r in records],
        "AGNbol_unc": [r["AGNbol_unc"] for r in records],
        r"SFR [M$_{\odot}$]": [r["SFR"] for r in records],
        r"un-corr SFR [M$_{\odot}$]": [r["un-corr SFR"] for r in records],
        r"Supernova rate [yr$^{-1}$]": [
            round(G * SNr_IMF((1 - r["AGNbol"]) * 10 ** r["log(LIR)"], "Yarno NK"), 2)
            for r in records
        ],
        r"un-corr Supernova rate [yr$^{-1}$]": [
            round(G * SNr_IMF(10 ** r["log(LIR)"], "Yarno NK"), 2)
            for r in records
        ],
        r"Flux(TeV) [GeV cm$^{-2}$ s$^{-1}$]": [
            round(
                Flux(
                    1e3, R, v, n_ism, H, 4, p_max,
                    G * SNr_IMF((1 - r["AGNbol"]) * 10 ** r["log(LIR)"], "Yarno NK"),
                    r["D_L [Mpc]"],
                ),
                14,
            )
            for r in records
        ],
        r"Flux(TeV) no AGN [GeV cm$^{-2}$ s$^{-1}$]": [
            round(
                Flux(
                    1e3, R, v, n_ism, H, 4, p_max,
                    G * SNr_IMF(10 ** r["log(LIR)"], "Yarno NK"),
                    r["D_L [Mpc]"],
                ),
                14,
            )
            for r in records
        ],
    },
    index=[r["ID"] for r in records],
)

pd.set_option("display.max_rows", 500)
display(df)

In [ ]:
#df.to_csv('dataframe')